In [15]:
import pandas as pd

df = pd.read_csv("emails_raw.csv")

# 1. Check data file
print("===== Head =====")
print(df.head()) #xem vài dòng đầu của dataset

print("\n==== Columns =====")
print(df.columns)

print("\n====== Info ======")
df.info()

===== Head =====
                                                text  spam
0  Subject: naturally irresistible your corporate...     1
1  Subject: the stock trading gunslinger  fanny i...     1
2  Subject: unbelievable new homes made easy  im ...     1
3  Subject: 4 color printing special  request add...     1
4  Subject: do not have money , get software cds ...     1

==== Columns =====
Index(['text', 'spam'], dtype='str')

====== Info ======
<class 'pandas.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5728 non-null   str  
 1   spam    5728 non-null   int64
dtypes: int64(1), str(1)
memory usage: 89.6 KB


In [16]:
# 2. Remove missing data & duplicate emails
# Check missing data
print("===== Missing Data =====")
print(df.isnull().sum())
if df.isnull().values.any():
    df = df.dropna() #dùng để xoá dòng bị thiếu data

# Check duplicate emails
print("\n===== Duplicate emails =====")
print("Shape before:", df.shape)

print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

===== Missing Data =====
text    0
spam    0
dtype: int64

===== Duplicate emails =====
Shape before: (5728, 2)
Duplicate rows: 33
Shape after removing duplicates: (5695, 2)


In [17]:
# 3. Check label có cân bằng không (vì nếu quá lệch nhau => accuracy bị ảnh hưởng)
# Độ lệch 70:30 đến 85:15 là đẹp
counts = df["spam"].value_counts()

percent = df["spam"].value_counts(
    normalize=True
) * 100

print("Counts:")
print(counts)

print("\nPercentages:")
for label, p in percent.items():
    print(f"Class {label}: {p:.2f}%")

Counts:
spam
0    4327
1    1368
Name: count, dtype: int64

Percentages:
Class 0: 75.98%
Class 1: 24.02%


In [18]:
# 4. Text cleaning
import re

def clean(text):
    # Chuyển toàn bộ thành chữ thường
    text = text.lower()
    # Remove email reply prefixes (tiền tố)
    text = re.sub(
        r'((re|fw|fwd)\s*:)+',
        ' ',
        text
    )

    # remove "subject:"
    text = re.sub(
        r'subject\s*:',
        ' ',
        text
    )
    # Xoá URL
    text = re.sub(r'http\S+|www\S+',' ',text)
    # Xóa HTML tags
    text = re.sub(r'<.*?>',' ',text)
    # Xoá mọi thứ KHÔNG phải chữ cái
    text = re.sub(r'[^a-zA-Z ]',' ',text)
    # Dọn khoảng trắng thừa
    text = re.sub(r'\s+',' ',text).strip()

    return text

df["text"] = df["text"].apply(clean)
df.to_csv(
    "emails_cleaned.csv",
    index=False
)

# remove blank emails after cleaning
df = df[df["text"].str.strip() != ""]

# remove duplicates after cleaning
df = df.drop_duplicates(subset=["text"])

# Check
print("Blank rows:",
      (df["text"] == "").sum())

print("Duplicate texts:",
      df["text"].duplicated().sum())
df.to_csv("emails_cleaned.csv", index=False)

Blank rows: 0
Duplicate texts: 0


In [ ]:
# 5. Chia train / test
from sklearn.model_selection import train_test_split
X = df["text"]
y = df["spam"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42, #cố định cách chia
    stratify=y #cố định tỉ lệ spam/ham ở train và test giống tỉ lệ của dataset
)
print("Train samples:", len(X_train_text))
print("Test samples:", len(X_test_text))

# Lưu train / test chưa vector
train_df = pd.DataFrame({
    "text": X_train_text.values,   #.values để tránh mismatch index (tránh tạo dataframe với index lung tung)
    "spam": y_train.values
})

test_df = pd.DataFrame({
    "text": X_test_text.values,
    "spam": y_test.values
})

train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

Train samples: 4396
Test samples: 1100


In [25]:
# Count unique words before vectorization

all_words = " ".join(
    df["text"]
).split()

unique_words = set(
    all_words
)

print(
    "Total words:",
    len(all_words)
)

print(
    "Unique words:",
    len(unique_words)
)

Total words: 1237069
Unique words: 33727


In [ ]:
# 6. Vectorize: Chuyển text → số (Vì ML ko hiểu chữ)
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    min_df=2, #document frequency: chỉ giữ từ xuất hiện ít nhất 2 emails
    ngram_range=(1,2)
)
X_train = vectorizer.fit_transform(X_train_text)

X_test = vectorizer.transform(X_test_text)
# in shape
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Lưu vectorizer
joblib.dump(
    vectorizer,
    "tfidf_vectorizer.pkl"
)
print("Vectorizer saved!")

# lưu feature count để debug
print(
    "Vocabulary size:",
    len(vectorizer.vocabulary_)
)

# lưu data sau khi vectorize
joblib.dump(X_train, "X_train.pkl")
joblib.dump(X_test, "X_test.pkl")

joblib.dump(y_train, "y_train.pkl")
joblib.dump(y_test, "y_test.pkl")

Train shape: (4396, 10000)
Test shape: (1100, 10000)
Vectorizer saved!
Vocabulary size: 10000


['y_test.pkl']

In [ ]:
# (check blank)
df_check = pd.read_csv("emails_cleaned.csv")

print("Original:", df.shape)
print("Saved CSV:", df_check.shape)

Original: (5496, 2)
Saved CSV: (5496, 2)
